# FidelProbe (standalone)

Same evaluation as the repo, with all the code in this notebook.

## Setup

In [ ]:
!pip -q install -U "transformers>=4.57" accelerate "huggingface_hub>=1.6"
!apt-get -qq install -y fonts-noto-core fonts-noto-cjk fonts-sil-abyssinica > /dev/null
!mkdir -p /content/fidelprobe/fidelprobe
%cd /content/fidelprobe/fidelprobe
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

### `scripts.py`

In [ ]:
%%writefile scripts.py
"""Helpers for splitting Ethiopic and Hangul syllables into consonant + vowel.

Unicode stores Ethiopic in rows of 8 (seven vowel orders, then the labialised
form), so row = consonant and column = vowel. Hangul is 0xAC00 + (lead*21 + vowel)*28 + tail.
"""

ETH_START, ETH_END = 0x1200, 0x135A  # punctuation starts at 0x1360

# the 33 Amharic base consonants (1st order); 33 x 7 = 231 core characters
ETH_BASES = list("ሀለሐመሠረሰሸቀበተቸኀነኘአከኸወዐዘዠየደጀገጠጨጰጸፀፈፐ")
assert len(ETH_BASES) == 33

# ግዕዝ ካዕብ ሣልስ ራብዕ ኃምስ ሳድስ ሳብዕ
ETH_ORDERS = ["ä", "u", "i", "a", "e", "ə", "o"]
ETH_ORDER_NAMES = ["gəʿəz", "kaʿəb", "saləs", "rabəʿ", "haməs", "sadəs", "sabəʿ"]


def eth_compose(base: str, order: int) -> str:
    """base is the 1st-order form, order is 1-7."""
    return chr(ord(base) + order - 1)


def eth_core_chars():
    """All 231 core characters as (char, base, order)."""
    return [(eth_compose(b, o), b, o) for b in ETH_BASES for o in range(1, 8)]


def is_ethiopic(ch: str) -> bool:
    return ETH_START <= ord(ch) < ETH_END


HANGUL_LEADS = "ㄱㄲㄴㄷㄸㄹㅁㅂㅃㅅㅆㅇㅈㅉㅊㅋㅌㅍㅎ"
# seven vowels so the Korean control has the same 7-way choice as Fidel
HANGUL_VOWELS = [(0, "ㅏ", "a"), (4, "ㅓ", "eo"), (8, "ㅗ", "o"), (13, "ㅜ", "u"),
                 (18, "ㅡ", "eu"), (20, "ㅣ", "i"), (1, "ㅐ", "ae")]


def is_hangul(ch: str) -> bool:
    return 0xAC00 <= ord(ch) <= 0xD7A3


def hangul_compose(lead: int, vowel: int, tail: int = 0) -> str:
    return chr(0xAC00 + (lead * 21 + vowel) * 28 + tail)


def hangul_core_chars():
    """19 consonants x 7 vowels, no final consonant."""
    out = []
    for li, lj in enumerate(HANGUL_LEADS):
        for k, (vi, _, _) in enumerate(HANGUL_VOWELS):
            out.append((hangul_compose(li, vi), lj, k + 1))
    return out


def decompose(ch: str):
    """(script, consonant, vowel) for a syllable, None for anything else.

    For Hangul the "consonant" is lead + final, so two syllables that only
    differ in the vowel count as a vowel-form mistake.
    """
    cp = ord(ch)
    if is_ethiopic(ch):
        off = cp - ETH_START
        return ("eth", off // 8, off % 8)
    if is_hangul(ch):
        off = cp - 0xAC00
        lead, rem = divmod(off, 21 * 28)
        vowel, tail = divmod(rem, 28)
        return ("hangul", (lead, tail), vowel)
    return None


def substitution_type(ref: str, hyp: str) -> str:
    """What kind of mistake is reading ref as hyp?"""
    if ref == hyp:
        return "correct"
    dr, dh = decompose(ref), decompose(hyp)
    if dr is None:
        return "other"
    if dh is None and 0x3131 <= ord(hyp) <= 0x318E:
        return "fragment"  # a lone jamo like ㅡ instead of a full syllable
    if dh is None or dh[0] != dr[0]:
        return "script_escape"  # wrote Latin, Greek, Georgian, ...
    if dr[1] == dh[1]:
        return "vowel_form"
    if dr[2] == dh[2]:
        return "consonant"
    return "both"


def align(ref: str, hyp: str):
    """Plain Levenshtein alignment -> [(op, ref_char, hyp_char), ...]"""
    n, m = len(ref), len(hyp)
    d = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        d[i][0] = i
    for j in range(m + 1):
        d[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            d[i][j] = min(d[i - 1][j] + 1, d[i][j - 1] + 1,
                          d[i - 1][j - 1] + (ref[i - 1] != hyp[j - 1]))
    ops, i, j = [], n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and d[i][j] == d[i - 1][j - 1] + (ref[i - 1] != hyp[j - 1]):
            ops.append(("eq" if ref[i - 1] == hyp[j - 1] else "sub", ref[i - 1], hyp[j - 1]))
            i, j = i - 1, j - 1
        elif i > 0 and d[i][j] == d[i - 1][j] + 1:
            ops.append(("del", ref[i - 1], ""))
            i -= 1
        else:
            ops.append(("ins", "", hyp[j - 1]))
            j -= 1
    return ops[::-1]


def error_breakdown(ref: str, hyp: str):
    """Count edits, splitting substitutions by vowel / consonant / script."""
    counts = {"ref_len": len(ref), "eq": 0, "del": 0, "ins": 0, "vowel_form": 0,
              "consonant": 0, "both": 0, "script_escape": 0, "fragment": 0, "other": 0}
    for op, r, h in align(ref, hyp):
        if op == "sub":
            counts[substitution_type(r, h)] += 1
        else:
            counts[op] += 1
    return counts


### `build_dataset.py`

In [ ]:
%%writefile build_dataset.py
"""Builds the probe set: probes.jsonl plus the rendered images.

Gold answers come from the Unicode layout, so nothing is hand-labelled.
Each Ethiopic task has a Korean twin to compare against.
"""
import argparse
import json
import random
from pathlib import Path

from PIL import Image, ImageDraw, ImageFont

from scripts import (ETH_BASES, HANGUL_LEADS, HANGUL_VOWELS, decompose,
                     eth_compose, eth_core_chars, hangul_compose,
                     hangul_core_chars, is_ethiopic)

ETH_FONT = "/usr/share/fonts/truetype/noto/NotoSansEthiopic-Regular.ttf"
ETH_FONT_ALT = "/usr/share/fonts/truetype/abyssinica/AbyssinicaSIL-Regular.ttf"
KOR_FONT = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"

# everyday Amharic words
AMHARIC_WORDS = """ሰላም ኢትዮጵያ አዲስ አበባ ቤት ውሃ ዳቦ ቡና ትምህርት መጽሐፍ ልጅ እናት አባት ወንድም
እህት ሀገር ከተማ መንገድ መኪና ገበያ ገንዘብ ስራ ጊዜ ቀን ሌሊት ፀሐይ ጨረቃ ኮከብ ዝናብ ተራራ
ወንዝ ባሕር ዛፍ ፍቅር ሰው ሴት ወንድ ቤተሰብ ምግብ እንጀራ ወጥ ሻይ ወተት ማር ጨው በርበሬ
ሽንኩርት ዶሮ በግ ላም ፈረስ ውሻ ድመት ወፍ አሳ ታሪክ ሙዚቃ ዘፈን ጨዋታ ኳስ ተማሪ መምህር
ሐኪም ሆስፒታል ቢሮ ስልክ ደብዳቤ ጥያቄ መልስ ቁጥር ፊደል ቃል ዓመት ወር ሳምንት ዛሬ ነገ
ትናንት ጥሩ መጥፎ ትልቅ ትንሽ ቆንጆ ሞቃት ቀዝቃዛ አመሰግናለሁ እንኳን ደህና ነጻነት ሰላምታ
ዩኒቨርሲቲ ቤተክርስቲያን መስጊድ ቴሌቪዥን ሬዲዮ ጋዜጣ""".split()

KOREAN_WORDS = """안녕 사랑 학교 친구 가족 어머니 아버지 나무 바다 하늘 사람 시간 음식 김치
한국 서울 도시 자동차 컴퓨터 전화 노래 음악 영화 책 공부 선생님 학생 병원 의사 물 우유
커피 고양이 강아지 새 꽃 봄 여름 가을 겨울 아침 저녁 오늘 내일 어제 좋다 나쁘다 크다
작다 감사합니다 미안합니다 대학교 도서관 시장 돈 일 이름 나라 언어 역사 바람 구름 비
눈 산 강 길 문 집 방 창문 의자 신문 편지 질문 대답 숫자 글자 단어 주 달 해""".split()

# short sentences for the line OCR task
AMHARIC_LINES = [
    "ሰላም እንዴት ነህ?", "አዲስ አበባ የኢትዮጵያ ዋና ከተማ ናት።", "ቡና መጠጣት እወዳለሁ።",
    "ልጆቹ ትምህርት ቤት ሄዱ።", "ዛሬ ዝናብ ይዘንባል።", "እናቴ እንጀራ ጋገረች።",
    "መጽሐፉን አነበብኩት።", "ነገ ገበያ እንሄዳለን።", "ውሻው በግቢው ውስጥ ይጫወታል።",
    "ፀሐይ በምሥራቅ ትወጣለች።", "አባቴ መኪና ይነዳል።", "ተማሪዎቹ ፈተና ወሰዱ።",
    "ሐኪሙ ታማሚውን አከመ።", "ወንዙ በጣም ትልቅ ነው።", "ስልኬ ጠፋብኝ።",
    "እህቴ ዘፈን ትዘፍናለች።", "ጓደኛዬ ደብዳቤ ጻፈልኝ።", "ቤተሰቤ በጣም ትልቅ ነው።",
    "ሻይ ወይስ ቡና ትፈልጋለህ?", "የአማርኛ ፊደል ብዙ ቁምፊዎች አሉት።", "ዶሮዎቹ በጠዋት ይጮኻሉ።",
    "መምህሩ ጥያቄ ጠየቀ።", "ገንዘቤን በባንክ አስቀመጥኩ።", "በዓሉ በደስታ ተከበረ።",
    "ሙዚቃ ማዳመጥ ያስደስተኛል።", "ተራራውን ወጣን።", "ጨረቃዋ ዛሬ ሙሉ ናት።",
    "ኳስ መጫወት እንወዳለን።", "ቀኑ በጣም ሞቃት ነው።", "አመሰግናለሁ ደህና ሁን።",
]

KOREAN_LINES = [
    "안녕하세요 만나서 반갑습니다.", "서울은 한국의 수도입니다.", "저는 커피를 좋아합니다.",
    "아이들이 학교에 갔습니다.", "오늘 비가 옵니다.", "어머니가 김치를 만드셨다.",
    "그 책을 다 읽었습니다.", "내일 시장에 갑니다.", "강아지가 마당에서 놉니다.",
    "해가 동쪽에서 뜹니다.", "아버지가 차를 운전합니다.", "학생들이 시험을 봤습니다.",
    "의사가 환자를 치료했다.", "그 강은 아주 큽니다.", "전화기를 잃어버렸어요.",
    "언니가 노래를 부릅니다.", "친구가 편지를 써 주었다.", "우리 가족은 대가족입니다.",
    "차와 커피 중 무엇을 원하세요?", "한글은 자음과 모음으로 이루어져 있다.",
    "닭이 아침에 웁니다.", "선생님이 질문을 했다.", "은행에 돈을 맡겼습니다.",
    "명절을 즐겁게 보냈다.", "음악 듣는 것을 좋아해요.", "우리는 산에 올랐다.",
    "오늘 밤 달이 둥글다.", "축구 하는 것을 좋아합니다.", "날씨가 매우 덥습니다.",
    "감사합니다 안녕히 가세요.",
]

ETH_SCHEME = ("The Ethiopic (Ge'ez/Fidel) script writes each syllable as a consonant in one of "
              "seven vowel orders: 1st=ä, 2nd=u, 3rd=i, 4th=a, 5th=e, 6th=ə (or no vowel), 7th=o. "
              "For example, the consonant በ (b) has the forms በ(bä) ቡ(bu) ቢ(bi) ባ(ba) ቤ(be) ብ(bə) ቦ(bo).")
KOR_SCHEME = ("The Korean Hangul script writes each syllable as a consonant combined with a vowel. "
              "In this question we number seven vowels as: 1st=ㅏ(a), 2nd=ㅓ(eo), 3rd=ㅗ(o), 4th=ㅜ(u), "
              "5th=ㅡ(eu), 6th=ㅣ(i), 7th=ㅐ(ae). For example, the consonant ㅂ (b) has the forms "
              "바(ba) 버(beo) 보(bo) 부(bu) 브(beu) 비(bi) 배(bae).")
ORD = ["1st", "2nd", "3rd", "4th", "5th", "6th", "7th"]
ONE_CHAR = "Answer with a single character only."


def kor_order_char(lead_jamo, order):
    return hangul_compose(HANGUL_LEADS.index(lead_jamo), HANGUL_VOWELS[order - 1][0])


def text_char_probes(rng):
    items = []
    sets = [("eth", eth_core_chars(), ETH_SCHEME), ("hangul", hangul_core_chars(), KOR_SCHEME)]
    for script, chars, scheme in sets:
        for ch, base, order in chars:
            meta = {"script": script, "char": ch, "base": base, "order": order}
            base_word = "1st-order (base) form" if script == "eth" else "consonant letter (jamo)"
            items.append({**meta, "task": "decompose_consonant",
                          "prompt": f"{scheme}\n\nWhat is the base consonant of the syllable {ch}? "
                                    f"Give its {base_word}. {ONE_CHAR}",
                          "gold": base, "answer_type": "char"})
            items.append({**meta, "task": "identify_order",
                          "prompt": f"{scheme}\n\nWhich vowel order (1-7) is the syllable {ch}? "
                                    "Answer with a single digit only.",
                          "gold": str(order), "answer_type": "digit"})
            tgt = rng.choice([o for o in range(1, 8) if o != order])
            compose = eth_compose(base, tgt) if script == "eth" else kor_order_char(base, tgt)
            items.append({**meta, "task": "compose", "target_order": tgt,
                          "prompt": f"{scheme}\n\nWrite the {ORD[tgt - 1]}-order syllable of the "
                                    f"consonant {base}. {ONE_CHAR}",
                          "gold": compose, "answer_type": "char"})
            items.append({**meta, "task": "transform", "target_order": tgt,
                          "prompt": f"{scheme}\n\nKeep the consonant of the syllable {ch} but change it "
                                    f"to the {ORD[tgt - 1]} vowel order. {ONE_CHAR}",
                          "gold": compose, "answer_type": "char"})
    return items


def flip(word, script, kind, rng):
    """Swap the vowel or the consonant of one random syllable in the word."""
    base_rows = [(ord(b) - 0x1200) // 8 for b in ETH_BASES]
    idx = [i for i, c in enumerate(word) if decompose(c) and
           (script != "eth" or ((ord(c) - 0x1200) // 8 in base_rows and (ord(c) - 0x1200) % 8 < 7))]
    if not idx:
        return None
    i = rng.choice(idx)
    c = word[i]
    if script == "eth":
        off = ord(c) - 0x1200
        row, v = divmod(off, 8)
        if kind == "vowel":
            new = chr(0x1200 + row * 8 + rng.choice([o for o in range(7) if o != v]))
        else:
            new = chr(0x1200 + rng.choice([r for r in base_rows if r != row]) * 8 + v)
    else:
        off = ord(c) - 0xAC00
        lead, rem = divmod(off, 21 * 28)
        vowel, tail = divmod(rem, 28)
        if kind == "vowel":
            new = hangul_compose(lead, rng.choice([x for x in range(21) if x != vowel]), tail)
        else:
            new = hangul_compose(rng.choice([x for x in range(19) if x != lead]), vowel, tail)
    return word[:i] + new + word[i + 1:]


def minimal_pair_probes(rng):
    items = []
    for script, words in [("eth", AMHARIC_WORDS), ("hangul", KOREAN_WORDS)]:
        for w in words:
            for cond in ["identical", "vowel", "consonant"]:
                w2 = w if cond == "identical" else flip(w, script, cond, rng)
                if w2 is None:
                    continue
                a, b = (w, w2) if rng.random() < 0.5 else (w2, w)
                items.append({"script": script, "task": "minimal_pair", "condition": cond,
                              "word_a": a, "word_b": b,
                              "prompt": "Are these two words spelled exactly the same, character for "
                                        f"character?\nWord A: {a}\nWord B: {b}\nAnswer yes or no.",
                              "gold": "yes" if cond == "identical" else "no",
                              "answer_type": "yesno"})
    return items


def render(text, font_path, size, out_path, pad=24):
    font = ImageFont.truetype(font_path, size)
    l, t, r, b = ImageDraw.Draw(Image.new("L", (1, 1))).textbbox((0, 0), text, font=font)
    img = Image.new("RGB", (r - l + 2 * pad, b - t + 2 * pad), "white")
    ImageDraw.Draw(img).text((pad - l, pad - t), text, font=font, fill="black")
    img.save(out_path)


def visual_probes(out):
    img_dir = out / "images"
    img_dir.mkdir(parents=True, exist_ok=True)
    items = []
    for script, chars, font in [("eth", eth_core_chars(), ETH_FONT),
                                ("hangul", hangul_core_chars(), KOR_FONT)]:
        for ch, base, order in chars:
            name = f"char_{script}_{ord(ch):04X}.png"
            render(ch, font, 64, img_dir / name)
            items.append({"script": script, "task": "ocr_char", "char": ch, "base": base,
                          "order": order, "image": f"images/{name}",
                          "prompt": "What single character is shown in this image? "
                                    "Reply with only that character.",
                          "gold": ch, "answer_type": "text"})
    for script, words, lines, font in [("eth", AMHARIC_WORDS, AMHARIC_LINES, ETH_FONT),
                                       ("hangul", KOREAN_WORDS, KOREAN_LINES, KOR_FONT)]:
        for i, w in enumerate(words):
            name = f"word_{script}_{i:03d}.png"
            render(w, font, 48, img_dir / name)
            items.append({"script": script, "task": "ocr_word", "image": f"images/{name}",
                          "prompt": "Transcribe the text in this image exactly. "
                                    "Reply with only the transcription.",
                          "gold": w, "answer_type": "text"})
        for i, s in enumerate(lines):
            name = f"line_{script}_{i:03d}.png"
            render(s, font, 40, img_dir / name)
            items.append({"script": script, "task": "ocr_line", "image": f"images/{name}",
                          "prompt": "Transcribe the text in this image exactly. "
                                    "Reply with only the transcription.",
                          "gold": s, "answer_type": "text"})
    return items


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="data")
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args()
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    rng = random.Random(args.seed)
    items = text_char_probes(rng) + minimal_pair_probes(rng) + visual_probes(out)
    for i, it in enumerate(items):
        it["id"] = f"{it['task']}-{it['script']}-{i:05d}"
    with open(out / "probes.jsonl", "w", encoding="utf-8") as f:
        for it in items:
            f.write(json.dumps(it, ensure_ascii=False) + "\n")
    from collections import Counter
    print(Counter((it["task"], it["script"]) for it in items))
    assert all(is_ethiopic(c) or not c.strip() or c in "?።" for w in AMHARIC_WORDS for c in w)


if __name__ == "__main__":
    main()


### `run_eval.py`

In [ ]:
%%writefile run_eval.py
"""Runs a VLM over probes.jsonl with greedy decoding.

Answers are appended as they come, so a crashed run picks up where it stopped.
"""
import argparse
import json
import time
from pathlib import Path

import torch
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor

SYSTEM = ("You are taking a test. Reply with the final answer only, exactly in the requested "
          "format. Do not explain, do not repeat the question.")
MAX_NEW = {"char": 12, "digit": 6, "yesno": 4, "text": 96}


def load(model_id):
    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForImageTextToText.from_pretrained(
        model_id, dtype=dtype, device_map="auto" if torch.cuda.is_available() else None)
    proc = AutoProcessor.from_pretrained(model_id)
    model.eval()
    return model, proc


@torch.inference_mode()
def generate(model, proc, item, data_dir):
    content = []
    if item.get("image"):
        content.append({"type": "image", "image": Image.open(data_dir / item["image"]).convert("RGB")})
    content.append({"type": "text", "text": item["prompt"]})
    msgs = [{"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
            {"role": "user", "content": content},
            {"role": "assistant", "content": [{"type": "text", "text": "Answer:"}]}]
    # without the "Answer:" prefill the 2B model tends to restate the question
    inputs = proc.apply_chat_template(msgs, tokenize=True, continue_final_message=True,
                                      return_dict=True, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=MAX_NEW[item["answer_type"]], do_sample=False)
    return proc.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="Qwen/Qwen3-VL-2B-Instruct")
    ap.add_argument("--data", default="../data")
    ap.add_argument("--out", default="../results")
    ap.add_argument("--tasks", default="", help="comma-separated subset of tasks")
    ap.add_argument("--limit", type=int, default=0)
    args = ap.parse_args()

    data_dir = Path(args.data)
    items = [json.loads(l) for l in open(data_dir / "probes.jsonl", encoding="utf-8")]
    if args.tasks:
        keep = set(args.tasks.split(","))
        items = [it for it in items if it["task"] in keep]
    if args.limit:
        items = items[: args.limit]

    out_dir = Path(args.out)
    out_dir.mkdir(parents=True, exist_ok=True)
    pred_path = out_dir / f"predictions_{args.model.replace('/', '__')}.jsonl"
    done = set()
    if pred_path.exists():
        done = {json.loads(l)["id"] for l in open(pred_path, encoding="utf-8")}
    todo = [it for it in items if it["id"] not in done]
    print(f"{len(items)} items, {len(todo)} to run", flush=True)

    torch.manual_seed(0)
    model, proc = load(args.model)
    t0 = time.time()
    with open(pred_path, "a", encoding="utf-8") as f:
        for n, it in enumerate(todo, 1):
            pred = generate(model, proc, it, data_dir)
            f.write(json.dumps({"id": it["id"], "model": args.model, "pred": pred},
                               ensure_ascii=False) + "\n")
            f.flush()
            if n % 25 == 0:
                rate = (time.time() - t0) / n
                print(f"{n}/{len(todo)}  {rate:.1f}s/item  eta {rate * (len(todo) - n) / 60:.0f} min",
                      flush=True)


if __name__ == "__main__":
    main()


### `tokenizer_probe.py`

In [ ]:
%%writefile tokenizer_probe.py
"""Checks how the tokenizer splits Fidel characters.

If the seven forms of a consonant never share a token, the model has no
direct way to tell from its input that ቡ and ቦ are the same consonant.
"""
import argparse
import json
from collections import defaultdict
from pathlib import Path

from transformers import AutoTokenizer

from build_dataset import AMHARIC_LINES, KOREAN_LINES
from scripts import eth_core_chars, hangul_core_chars

ENGLISH_LINES = [  # translations of the first few Amharic lines
    "Hello, how are you?", "Addis Ababa is the capital city of Ethiopia.", "I love drinking coffee.",
    "The children went to school.", "It will rain today.", "My mother baked injera.",
]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="Qwen/Qwen3-VL-2B-Instruct")
    ap.add_argument("--out", default="../results")
    args = ap.parse_args()
    tok = AutoTokenizer.from_pretrained(args.model)
    enc = lambda s: tok.encode(s, add_special_tokens=False)

    report = {"model": args.model}
    for script, chars in [("eth", eth_core_chars()), ("hangul", hangul_core_chars())]:
        ids = {ch: enc(ch) for ch, _, _ in chars}
        by_base = defaultdict(list)
        for ch, base, _ in chars:
            by_base[base].append(ids[ch])
        # does every form of this consonant share at least one token?
        shared = sum(bool(set.intersection(*[set(x) for x in forms])) for forms in by_base.values())
        report[script] = {
            "n_chars": len(chars),
            "single_token_chars": sum(len(v) == 1 for v in ids.values()),
            "mean_tokens_per_char": sum(map(len, ids.values())) / len(ids),
            "consonants_with_shared_token_across_forms": f"{shared}/{len(by_base)}",
            "example": {ch: [tok.convert_ids_to_tokens(i) for i in [ids[ch]]][0]
                        for ch, _, _ in chars[:7]},
        }
    for name, lines in [("eth", AMHARIC_LINES), ("hangul", KOREAN_LINES), ("english", ENGLISH_LINES)]:
        n_tok = sum(len(enc(s)) for s in lines)
        n_chr = sum(len(s) for s in lines)
        report.setdefault("lines", {})[name] = {"tokens_per_char": n_tok / n_chr,
                                                "tokens_per_line": n_tok / len(lines)}
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    tag = args.model.replace("/", "__")
    (out / f"tokenizer_{tag}.json").write_text(json.dumps(report, indent=2, ensure_ascii=False))
    print(json.dumps(report, indent=2, ensure_ascii=False))


if __name__ == "__main__":
    main()


### `analyze.py`

In [ ]:
%%writefile analyze.py
"""Scores the predictions and writes summary json, a markdown table and plots."""
import argparse
import json
import re
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from scripts import decompose, error_breakdown, is_ethiopic, is_hangul, substitution_type

SCRIPT_TEST = {"eth": is_ethiopic, "hangul": is_hangul}
LABEL = {"eth": "Ethiopic (Amharic)", "hangul": "Hangul (Korean control)"}
ERR_KEYS = ["vowel_form", "consonant", "both", "script_escape", "fragment", "other", "del", "ins"]


def extract(item, pred):
    t = item["answer_type"]
    if t == "digit":
        m = re.search(r"[1-7]", pred)
        return m.group(0) if m else ""
    if t == "yesno":
        m = re.search(r"\b(yes|no)\b", pred.lower())
        return m.group(1) if m else ""
    if t == "char":
        # take the first char in the expected script, otherwise whatever came first
        same = [c for c in pred if SCRIPT_TEST[item["script"]](c) or c in "ㄱㄲㄴㄷㄸㄹㅁㅂㅃㅅㅆㅇㅈㅉㅊㅋㅌㅍㅎ"]
        if same:
            return same[0]
        rest = [c for c in pred if not c.isspace()]
        return rest[0] if rest else ""
    return " ".join(pred.split())


def score(items, preds):
    rows = []
    for it in items:
        if it["id"] not in preds:
            continue
        raw = preds[it["id"]]
        ans = extract(it, raw)
        row = {**it, "raw": raw, "answer": ans, "correct": ans == it["gold"]}
        if it["task"] in ("compose", "transform", "ocr_char") or (
                it["task"] == "decompose_consonant" and it["script"] == "eth"):
            row["err_type"] = substitution_type(it["gold"], ans) if ans else "empty"
        if it["task"] in ("ocr_word", "ocr_line"):
            g = " ".join(it["gold"].split())
            row["breakdown"] = error_breakdown(g, ans)
        rows.append(row)
    return rows


def summarize(rows):
    S = defaultdict(dict)
    by = defaultdict(list)
    for r in rows:
        by[(r["task"], r["script"])].append(r)
    for (task, script), rs in sorted(by.items()):
        d = {"n": len(rs), "accuracy": sum(r["correct"] for r in rs) / len(rs)}
        if "err_type" in rs[0]:
            errs = Counter(r["err_type"] for r in rs if not r["correct"])
            d["errors"] = dict(errs)
            wrong = sum(errs.values())
            d["vowel_form_share_of_errors"] = errs.get("vowel_form", 0) / wrong if wrong else 0.0
        if task == "minimal_pair":
            for cond in ["identical", "vowel", "consonant"]:
                cs = [r for r in rs if r["condition"] == cond]
                if cs:
                    d[f"acc_{cond}"] = sum(r["correct"] for r in cs) / len(cs)
        if "breakdown" in rs[0]:
            tot = Counter()
            for r in rs:
                tot.update(r["breakdown"])
            edits = sum(tot[k] for k in ERR_KEYS)
            subs = sum(tot[k] for k in ["vowel_form", "consonant", "both", "script_escape", "fragment", "other"])
            d["cer"] = edits / tot["ref_len"]
            d["edits"] = {k: tot[k] for k in ERR_KEYS}
            d["vowel_form_share_of_substitutions"] = tot["vowel_form"] / subs if subs else 0.0
            d["exact_match"] = d.pop("accuracy")
        if task in ("ocr_char", "identify_order", "transform", "compose"):
            key = "target_order" if task in ("compose", "transform") else "order"
            per = defaultdict(list)
            for r in rs:
                per[r[key]].append(r["correct"])
            d["acc_by_order"] = {k: sum(v) / len(v) for k, v in sorted(per.items())}
        S[task][script] = d
    return S


def fig_task_bars(S, path):
    tasks = [t for t in ["ocr_char", "decompose_consonant", "identify_order", "compose", "transform",
                         "minimal_pair"] if t in S]
    fig, ax = plt.subplots(figsize=(9, 3.8))
    w = 0.38
    for k, (script, color) in enumerate([("eth", "#c2410c"), ("hangul", "#1d4ed8")]):
        vals = [S[t].get(script, {}).get("accuracy", 0) for t in tasks]
        bars = ax.bar([i + (k - 0.5) * w for i in range(len(tasks))], vals, w, label=LABEL[script],
                      color=color)
        ax.bar_label(bars, fmt="%.2f", fontsize=8)
    chance = {"identify_order": 1 / 7, "minimal_pair": 0.5}
    for i, t in enumerate(tasks):
        if t in chance:
            ax.hlines(chance[t], i - 0.45, i + 0.45, colors="gray", linestyles=":", lw=1)
    ax.set_xticks(range(len(tasks)), [t.replace("_", "\n") for t in tasks], fontsize=9)
    ax.set_ylim(0, 1.08)
    ax.set_ylabel("accuracy")
    ax.legend(frameon=False, fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def fig_order_confusion(rows, path):
    """Vowel-order confusion for single-character OCR, only where the consonant was right."""
    M = [[0] * 7 for _ in range(7)]
    for r in rows:
        if r["task"] != "ocr_char" or r["script"] != "eth" or not r["answer"]:
            continue
        g, a = decompose(r["gold"]), decompose(r["answer"])
        if a and a[0] == "eth" and a[1] == g[1] and a[2] < 7:
            M[g[2]][a[2]] += 1
    fig, ax = plt.subplots(figsize=(4.4, 3.9))
    im = ax.imshow(M, cmap="Oranges")
    for i in range(7):
        for j in range(7):
            if M[i][j]:
                ax.text(j, i, M[i][j], ha="center", va="center", fontsize=8,
                        color="white" if M[i][j] > max(map(max, M)) / 2 else "black")
    lab = ["1 ä", "2 u", "3 i", "4 a", "5 e", "6 ə", "7 o"]
    ax.set_xticks(range(7), lab, fontsize=8)
    ax.set_yticks(range(7), lab, fontsize=8)
    ax.set_xlabel("predicted order (correct consonant)")
    ax.set_ylabel("true order")
    fig.colorbar(im, fraction=0.046)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def fig_error_mix(S, path):
    cats = [("vowel_form", "#c2410c"), ("consonant", "#1d4ed8"), ("both", "#7c3aed"),
            ("script_escape", "#6b7280"), ("del", "#a3a3a3"), ("ins", "#d4d4d4")]
    bars = [(f"{t.split('_')[1]}\n{s}", S[t][s]) for t in ["ocr_word", "ocr_line"] if t in S
            for s in ["eth", "hangul"] if s in S[t]]
    if not bars:
        return
    fig, ax = plt.subplots(figsize=(6.5, 3.6))
    for i, (name, d) in enumerate(bars):
        left = 0
        for c, col in cats:
            v = d["edits"].get(c, 0) / max(1, sum(d["edits"].values())) * d["cer"]
            ax.barh(i, v, left=left, color=col, label=c if i == 0 else None)
            left += v
        ax.text(left + 0.005, i, f"CER {d['cer']:.2f}", va="center", fontsize=8)
    ax.set_yticks(range(len(bars)), [b[0] for b in bars], fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel("character error rate, split by edit type")
    ax.legend(frameon=False, fontsize=8, ncol=3, loc="lower right")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def pct(x):
    return f"{100 * x:.1f}%"


def results_md(S, tag, tok=None):
    L = [f"### Results: `{tag.replace('__', '/')}`", ""]
    L += ["**Headline: Ethiopic vs the matched Hangul control (accuracy)**", "",
          "| Task | Ethiopic | Hangul | What it tests |", "|---|---|---|---|"]
    desc = {"ocr_char": "read one rendered character",
            "decompose_consonant": "name the base consonant of a character",
            "identify_order": "name the vowel order 1-7 (chance 14%)",
            "compose": "consonant + order -> character",
            "transform": "same consonant, new vowel order",
            "minimal_pair": "same/different spelling (chance 50%)"}
    for t, dsc in desc.items():
        if t in S:
            e, h = S[t].get("eth", {}), S[t].get("hangul", {})
            L.append(f"| {t} | {pct(e.get('accuracy', 0))} | {pct(h.get('accuracy', 0))} | {dsc} |")
    if "minimal_pair" in S:
        L += ["", "**Minimal pairs: accuracy by the kind of difference**", "",
              "| | identical | one vowel form changed | one consonant changed |", "|---|---|---|---|"]
        for s in ["eth", "hangul"]:
            d = S["minimal_pair"].get(s, {})
            L.append(f"| {LABEL[s]} | {pct(d.get('acc_identical', 0))} | {pct(d.get('acc_vowel', 0))} "
                     f"| {pct(d.get('acc_consonant', 0))} |")
    ocr = [(t, s) for t in ["ocr_word", "ocr_line"] if t in S for s in ["eth", "hangul"] if s in S[t]]
    if ocr:
        L += ["", "**Rendered-text OCR (character error rate, and what the errors are)**", "",
              "| Set | CER | exact match | vowel-form subs | consonant subs | other-script subs |"
              " deletions | insertions |", "|---|---|---|---|---|---|---|---|"]
        for t, s in ocr:
            d = S[t][s]
            e = d["edits"]
            L.append(f"| {t} / {s} | {pct(d['cer'])} | {pct(d['exact_match'])} | {e['vowel_form']} "
                     f"| {e['consonant']} | {e['script_escape']} | {e['del']} | {e['ins']} |")
    err_tasks = [t for t in ["ocr_char", "compose", "transform"] if t in S and "eth" in S[t]]
    if err_tasks:
        L += ["", "**Ethiopic single-character errors by type**", "",
              "| Task | vowel form (right consonant) | consonant | both | other script | n wrong |",
              "|---|---|---|---|---|---|"]
        for t in err_tasks:
            e = S[t]["eth"].get("errors", {})
            L.append(f"| {t} | {e.get('vowel_form', 0)} | {e.get('consonant', 0)} | {e.get('both', 0)} "
                     f"| {e.get('script_escape', 0)} | {sum(e.values())} |")
    if "ocr_char" in S and "eth" in S["ocr_char"]:
        bo = S["ocr_char"]["eth"]["acc_by_order"]
        L += ["", "**Ethiopic single-character OCR accuracy by vowel order**", "",
              "| " + " | ".join(f"{k} ({o})" for k, o in zip(range(1, 8), "ä u i a e ə o".split())) + " |",
              "|" + "---|" * 7, "| " + " | ".join(pct(bo.get(str(k), bo.get(k, 0))) for k in range(1, 8)) + " |"]
    if tok:
        L += ["", "**Tokenizer view of the Fidel grid**", "",
              "| | single-token chars | mean tokens / char | consonants whose 7 forms share a token |",
              "|---|---|---|---|"]
        for s in ["eth", "hangul"]:
            d = tok[s]
            L.append(f"| {LABEL[s]} | {d['single_token_chars']}/{d['n_chars']} | "
                     f"{d['mean_tokens_per_char']:.2f} | {d['consonants_with_shared_token_across_forms']} |")
        ln = tok["lines"]
        L.append("")
        L.append("Tokens per character on sentences: " + ", ".join(
            f"{k} {v['tokens_per_char']:.2f}" for k, v in ln.items()) + ".")
    L += ["", f"![task accuracy](results/fig_tasks_{tag}.png)",
          f"![vowel-order confusion](results/fig_order_confusion_{tag}.png)",
          f"![OCR errors](results/fig_ocr_errors_{tag}.png)"]
    return "\n".join(L) + "\n"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="../data")
    ap.add_argument("--results", default="../results")
    args = ap.parse_args()
    items = [json.loads(l) for l in open(Path(args.data) / "probes.jsonl", encoding="utf-8")]
    res = Path(args.results)
    for pf in sorted(res.glob("predictions_*.jsonl")):
        tag = pf.stem.replace("predictions_", "")
        preds = {}
        for l in open(pf, encoding="utf-8"):
            p = json.loads(l)
            preds[p["id"]] = p["pred"]
        rows = score(items, preds)
        S = summarize(rows)
        (res / f"summary_{tag}.json").write_text(json.dumps(S, indent=2, ensure_ascii=False))
        with open(res / f"scored_{tag}.jsonl", "w", encoding="utf-8") as f:
            for r in rows:
                r.pop("prompt", None)
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        fig_task_bars(S, res / f"fig_tasks_{tag}.png")
        fig_order_confusion(rows, res / f"fig_order_confusion_{tag}.png")
        fig_error_mix(S, res / f"fig_ocr_errors_{tag}.png")
        tok_path = res / f"tokenizer_{tag}.json"
        tok = json.loads(tok_path.read_text()) if tok_path.exists() else None
        (res / f"RESULTS_{tag}.md").write_text(results_md(S, tag, tok), encoding="utf-8")
        print(tag)
        print(json.dumps(S, indent=1, ensure_ascii=False))


if __name__ == "__main__":
    main()


### `upload_to_hf.py`

In [ ]:
%%writefile upload_to_hf.py
"""Creates the public HF bucket and uploads README, code, data, results and the model."""
import argparse
import shutil
import tempfile
from pathlib import Path

from huggingface_hub import HfApi, snapshot_download

ROOT = Path(__file__).resolve().parent.parent


def build_readme(dst):
    readme = (ROOT / "README.md").read_text(encoding="utf-8")
    tables = sorted((ROOT / "results").glob("RESULTS_*.md"))
    if tables and "<!-- RESULTS -->" in readme:
        readme = readme.replace("<!-- RESULTS -->",
                                "\n\n".join(t.read_text(encoding="utf-8") for t in tables))
    (dst / "README.md").write_text(readme, encoding="utf-8")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--bucket", default="fidelprobe", help="bucket name, or namespace/name")
    ap.add_argument("--model", default="Qwen/Qwen3-VL-2B-Instruct")
    ap.add_argument("--no-model", action="store_true", help="do not copy the model weights")
    args = ap.parse_args()

    api = HfApi()
    bucket_id = args.bucket if "/" in args.bucket else f"{api.whoami()['name']}/{args.bucket}"
    url = api.create_bucket(bucket_id, visibility="public", exist_ok=True)

    with tempfile.TemporaryDirectory() as tmp:
        stage = Path(tmp)
        build_readme(stage)
        for sub in ["fidelprobe", "data", "results"]:
            shutil.copytree(ROOT / sub, stage / sub,
                            ignore=shutil.ignore_patterns("__pycache__", "*.pyc"))
        if not args.no_model:
            snapshot_download(args.model, local_dir=stage / "model",
                              ignore_patterns=[".cache/*"])
            shutil.rmtree(stage / "model" / ".cache", ignore_errors=True)
        api.sync_bucket(str(stage), f"hf://buckets/{bucket_id}")
    print(f"Uploaded to https://huggingface.co/buckets/{bucket_id}  ({url})")


if __name__ == "__main__":
    main()


### `README.md`

In [ ]:
%%writefile ../README.md
---
license: cc-by-4.0
language: [am, ko]
tags: [evaluation, ethiopic, amharic, ocr, vision-language, tokenization, low-resource]
---

# FidelProbe: small frontier models read Ethiopic as a bag of opaque symbols

**Author:** Alefew Yimam (Fordham University)
**Model evaluated:** [`Qwen/Qwen3-VL-2B-Instruct`](https://huggingface.co/Qwen/Qwen3-VL-2B-Instruct) (2B parameters)
**Contents:** evaluation set (2,592 items + 602 rendered images), code, Colab notebook, raw and scored predictions, figures.

---

## 1. The blind spot: vowel-form blindness in the Ethiopic script

Amharic, spoken by more than 58 million people, is written in the Ethiopic script (*Fidel*, ፊደል).
Fidel is an **abugida**: each character is a consonant written in one of seven *vowel orders*.
The 33 core consonants × 7 orders give the 231 core characters every Ethiopian child learns as a grid:

| | ä | u | i | a | e | ə | o |
|---|---|---|---|---|---|---|---|
| **b** | በ | ቡ | ቢ | ባ | ቤ | ብ | ቦ |
| **s** | ሰ | ሱ | ሲ | ሳ | ሴ | ስ | ሶ |

Forms of one consonant differ only by a small stroke, ring or leg. In Amharic that small mark carries
grammar, not just pronunciation: **ሰበረ** "he broke", **ሰበሩ** "they broke" and **ስበር** "break!" differ only
in vowel orders. One wrong vowel form is a different word.

Two experiences led me to this gap.

* **As a reader and writer of Amharic.** When I use AI tools in Amharic, the output looks fluent
  but carries wrong vowel forms: the right consonant with the wrong order. A Latin-script reader
  would not notice. An Amharic reader sees broken grammar.
* **As a researcher on Amharic OCR.** My course project proposal (*A Controlled Architecture
  Comparison of CNN-LSTM, Attention, and Transformer Recognizers for the Fidel Amharic OCR Dataset*)
  builds on the Fidel benchmark (Chamisso et al., 2026). Its error analysis found that **most
  substitution errors across every model are the correct consonant with the wrong vowel form**,
  and it proposed no remedy. My proposal attacks this in dedicated OCR models with a *factored*
  output layer that predicts the consonant (33-way) and the vowel order (7-way) separately.

The question here is whether general-purpose frontier models, the ones people actually use, share
the same blind spot. My hypothesis: **they treat each Fidel character as an atomic, opaque symbol.
They know neither the consonant × vowel structure of the script nor how to see it.**

### Why standard benchmarks miss it

* Multilingual benchmarks such as FLORES, Belebele, Global-MMLU and AfriMMLU score *semantic*
  tasks at the sentence level. A model can pick the right multiple-choice answer while being unable
  to spell. The vowel-form structure is never tested directly.
* OCR and document benchmarks such as OCRBench, DocVQA and CC-OCR are dominated by Latin and CJK
  text. Where Ethiopic appears, results are aggregated into a single CER, and **CER does not
  distinguish a vowel-form slip from any other error**.
* Tokenizers hide the problem. Byte-level BPE assigns each Fidel character one or more opaque ids,
  so the model gets no signal that ቡ and ቦ share a consonant.
* There is no natural control. A low Amharic score is usually written off as "low-resource". The
  structural question needs a *matched* comparison against a script with the same
  consonant × vowel composition and plenty of data. Korean Hangul (바 버 보 부 …) is such a script.

---

## 2. Evaluation

### Model

[`Qwen/Qwen3-VL-2B-Instruct`](https://huggingface.co/Qwen/Qwen3-VL-2B-Instruct): a 2B-parameter
vision-language model from a current frontier open-weight family, trending on the Hub. It is
multilingual, does OCR, and is small enough to run on a free Colab T4 or even CPU. Decoding is greedy,
with an answer-only system prompt and an `Answer:` prefill so that the answer can be parsed reliably.

### Design: one structural question, three ways, with a matched control

Every gold answer is computed from Unicode arithmetic (Ethiopic: `0x1200 + 8·row + order`; Hangul:
`0xAC00 + (lead·21 + vowel)·28 + tail`), so scoring needs no human judgment. Every Ethiopic task has a
**Hangul control** with the same template, scheme description and answer format. Hangul is also a
consonant × vowel script, but far better represented in training data. The control separates
"the model has not learned this script's structure" from "syllabic scripts are hard".

| Probe | Items | Question |
|---|---|---|
| **Symbolic**: `decompose_consonant`, `identify_order`, `compose`, `transform` | 4 × (231 Ethiopic + 133 Hangul) | Does the model *know* the grid? The 7-order scheme is explained in every prompt with a worked example, so this tests applying it, not recalling it. |
| **Minimal pairs**: `minimal_pair` | 288 Ethiopic + 246 Hangul | Can it *notice* a single vowel-form change in a real word, compared with a single consonant change? |
| **Perception**: `ocr_char`, `ocr_word`, `ocr_line` | 231 + 96 + 30 Ethiopic; 133 + 82 + 30 Hangul | Can it *see* the difference? Rendered in Noto Sans, so there is no handwriting noise; this is a best-case upper bound. |
| **Tokenizer** | 231 + 133 characters | Is the consonant/vowel structure even visible in the input ids? |

OCR errors are aligned by edit distance. Every substitution is then classified as a **vowel-form error**
(same consonant, wrong order: the error type the Fidel benchmark found dominant), a consonant error,
both, or an **other-script** error (the model wrote Latin, Greek, Georgian or other characters).

### Reproduce

```bash
cd fidelprobe
python build_dataset.py --out ../data            # deterministic, seed 0
python run_eval.py --model Qwen/Qwen3-VL-2B-Instruct --data ../data --out ../results
python tokenizer_probe.py --out ../results
python analyze.py --data ../data --results ../results
```

Or open `fidelprobe/FidelProbe_colab.ipynb` in Colab (T4) and run all.

<!-- RESULTS -->

---

## 3. Path forward

The results point to three separable failures: the model cannot *see* vowel strokes, it does not
*know* the grid, and its tokenizer *hides* the grid. Each needs its own fix, and they compound.

### 3.1 Data curation: teach the grid explicitly, with vowel-form hard negatives

* **Fidel drills generated from Unicode arithmetic.** The grid is fully regular, so unlimited
  verified supervision is free: decompose, compose, transform, transliterate, and "what changed
  between ሰበረ and ሰበሩ?". These become instruction-tuning data, or verifiable rewards for RL
  (every answer is checkable by code, like math). This is the curriculum every Ethiopian
  first-grader goes through (ሀ ሁ ሂ ሃ ሄ ህ ሆ), and models have never been given it.
* **Vowel-form hard negatives for vision.** For every real Amharic word, render or collect its
  vowel-form neighbours: same consonants, one order changed, often another valid word. Train
  image–text contrastive and OCR objectives against them. Generic captioning data never forces a
  model to look at the small stroke that separates ሰ from ሱ, ሲ, ሳ, ሴ, ስ, ሶ.
* **Real handwriting and print.** The Fidel dataset (366k line images from 411 writers, CC-BY)
  plus multi-font rendering: Noto, Abyssinica SIL, Nyala and Power Geez fonts, with degradation.
  Today there is so little Ethiopic in VLM pre-training that the model falls back to
  similar-looking scripts. That is exactly the other-script error measured here.
* **Grammar-aware text data.** Vowel orders carry Amharic morphology (person, number, mood).
  Curate morphologically annotated text, with paradigm tables of verb roots across orders, so that
  the language model learns that a vowel change is a grammar change, not noise.

### 3.2 Architecture: make the consonant × vowel factorisation explicit

* **Factored character embeddings (input side).** Represent each Ethiopic character as
  `E_consonant[row] + E_order[order]` (plus a residual per-character embedding), instead of an
  opaque BPE id or a byte sequence split differently for each form. This is the input-side version
  of the **factored output layer** in my OCR proposal. It shares statistical strength across all 7
  forms of a consonant and makes a vowel change a single, learnable direction in embedding space.
  It also costs ~1 token per character instead of 1.5.
* **Factored output head.** Predict consonant (33-way) and order (7-way) as two heads whose
  log-probabilities are summed. This is the same idea as the Bengali.AI grapheme root / diacritic
  decomposition, applied inside an LLM or VLM decoder for Ethiopic code points. It could be a
  drop-in adapter on the LM head, restricted to the Ethiopic block.
* **Script-aware tokenizer surgery.** At minimum, ensure that all 231 core characters are single
  tokens (today only about half are) and initialise new token embeddings from consonant + order
  means. Byte-level fallback splits ሂ into a byte prefix shared across *different* consonants. That
  is the wrong grouping: the vowel order sits in the low 3 bits of the code point, so a
  structure-aware merge order is possible.
* **Higher-resolution glyph crops for the vision tower.** Vowel strokes are a few pixels wide.
  Dynamic-resolution encoders should up-weight small-glyph scripts, or use a glyph-level auxiliary
  head that predicts (consonant, order) per character region.

### 3.3 Fine-tuning paradigm

1. **Stage 1, structural SFT** (LoRA, cheap): Fidel drills plus vowel-form minimal pairs. Text-only,
   generated, verifiable.
2. **Stage 2, perceptual alignment:** OCR fine-tuning on Fidel plus rendered text, with an auxiliary
   factored (consonant, order) loss and a contrastive loss against vowel-form hard negatives.
3. **Stage 3, RL with verifiable rewards:** reward exact character match, with a larger penalty
   for vowel-form errors that change morphology.
4. **Evaluation:** report the **vowel-form confusion rate** alongside CER for every Ethiopic
   model. Every syllabic script (Ethiopic, Devanagari, Bengali, Kannada, Canadian Syllabics) needs a
   structural error breakdown like this one. A single CER hides the failure that matters most to
   readers.

---

## License and attribution

Code and generated probes: CC-BY-4.0. Model weights: Apache-2.0 (Qwen). Amharic and Korean word and
sentence lists were written for this benchmark. Related data: Fidel dataset (Chamisso et al., 2026),
`upanzi/fidel-dataset`.


## Build the probe set

In [ ]:
!python build_dataset.py --out ../data --seed 0

## Run the model

In [ ]:
MODEL = "Qwen/Qwen3-VL-2B-Instruct"
!python run_eval.py --model $MODEL --data ../data --out ../results

## Tokenizer check and scoring

In [ ]:
!python tokenizer_probe.py --model $MODEL --out ../results > /dev/null
!python analyze.py --data ../data --results ../results > /dev/null
from IPython.display import Image, Markdown, display
import glob
display(Markdown(open(glob.glob('../results/RESULTS_*.md')[0]).read().split('![')[0]))
for p in sorted(glob.glob('../results/fig_*.png')): display(Image(p))

## Some Ethiopic mistakes

In [ ]:
import pandas as pd
df = pd.read_json(glob.glob('../results/scored_*.jsonl')[0], lines=True)
df[(df.script == 'eth') & (~df.correct)][['task', 'gold', 'answer', 'raw']].sample(20, random_state=0)

## Upload to the Hugging Face bucket

In [ ]:
!python upload_to_hf.py --bucket fidelprobe --model $MODEL